In [1]:
import pandas as pd
import ast
from pathlib import Path

# =====================================================
# INPUT (ton chemin)
# =====================================================
movies_fr_final = pd.read_csv("../data/processed/movies_fr_final.csv", encoding="utf-8")

# =====================================================
# OUTPUT
# =====================================================
OUT_DIR = Path("../data/processed/powerbi")
OUT_DIR.mkdir(parents=True, exist_ok=True)



In [2]:
# =====================================================
# TABLE PRINCIPALE : on garde seulement les colonnes nécessaires
# =====================================================
KEEP_COLS = [
    "id_film",
    "titre_francais",
    "genres",
    "duree_minutes",
    "note_moyenne",
    "nombre_votes",
    "decennie",
    "noms_realisateurs",
    "principaux_acteurs",
    "societes_production",
    "pays_production",
    "langue_originale",
    "date_sortie",
    "budget",
    "recettes",
    "patrimonial",
    "zone_de_production",
    "annee_de_sortie",
    "genres_list",
]

main = movies_fr_final[KEEP_COLS].copy()

# Power BI : identifiant en texte + nom standardisé
main["id_film"] = main["id_film"].astype(str).str.strip()

# Numériques
NUM_COLS = ["duree_minutes", "note_moyenne", "nombre_votes", "budget", "recettes", "annee_de_sortie", "decennie"]
for c in NUM_COLS:
    main[c] = pd.to_numeric(main[c], errors="coerce")

# Date
main["date_sortie"] = pd.to_datetime(main["date_sortie"], errors="coerce").dt.date

# Bool patrimonial (tes valeurs sont "No"/"Yes" ou équivalent)
def to_bool(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    if s in {"1", "true", "vrai", "yes", "oui"}:
        return True
    if s in {"0", "false", "faux", "no", "non"}:
        return False
    return pd.NA

main["patrimonial"] = main["patrimonial"].apply(to_bool).astype("boolean")

# Export table principale
main.to_csv(OUT_DIR / "movies_fr_final_powerbi.csv", index=False, encoding="utf-8-sig")



In [3]:
# =====================================================
# TABLES BRIDGE (many-to-many)
# =====================================================
def to_list_cell(x):
    """Supporte: liste, NaN, 'A|B', 'A, B', \"['A','B']\"."""
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]

    s = str(x).strip()
    if not s:
        return []

    # Format liste python dans une string: "['A','B']"
    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return [str(v).strip() for v in parsed if str(v).strip()]
        except Exception:
            pass

    # fallback séparateurs
    if "|" in s:
        parts = s.split("|")
    elif "," in s:
        parts = s.split(",")
    else:
        parts = [s]

    return [p.strip() for p in parts if p.strip()]

def make_bridge(df_in, id_col, source_col, out_col):
    tmp = df_in[[id_col, source_col]].copy()
    tmp[source_col] = tmp[source_col].apply(to_list_cell)

    out = (
        tmp.explode(source_col)
           .rename(columns={source_col: out_col})
           .dropna(subset=[out_col])
    )

    out[id_col] = out[id_col].astype(str).str.strip()
    out[out_col] = out[out_col].astype("string").str.strip()

    out = out[out[out_col].notna() & (out[out_col] != "")]
    out = out.drop_duplicates()

    return out

# Genres (depuis genres_list)
table_genres = make_bridge(main, "id_film", "genres_list", "genre")

# Acteurs
table_acteurs = make_bridge(main, "id_film", "principaux_acteurs", "acteur")

# Réalisateurs
table_realisateurs = make_bridge(main, "id_film", "noms_realisateurs", "realisateur")

# Sociétés
table_societes = make_bridge(main, "id_film", "societes_production", "societe_production")

# Langue originale (dans ton CSV c’est une liste: ['anglais'])
table_langues = make_bridge(main, "id_film", "langue_originale", "langue_originale")

# Exports bridge
table_genres.to_csv(OUT_DIR / "tableBI_final_genres.csv", index=False, encoding="utf-8-sig")
table_acteurs.to_csv(OUT_DIR / "tableBI_final_acteurs.csv", index=False, encoding="utf-8-sig")
table_realisateurs.to_csv(OUT_DIR / "tableBI_final_realisateurs.csv", index=False, encoding="utf-8-sig")
table_societes.to_csv(OUT_DIR / "tableBI_final_societes_production.csv", index=False, encoding="utf-8-sig")
table_langues.to_csv(OUT_DIR / "tableBI_final_langues_originales.csv", index=False, encoding="utf-8-sig")

print(" OK - Exports générés dans :", OUT_DIR.resolve())
print(" - movies_fr_final_powerbi.csv (table principale)")
print(" - tableBI_final_genres.csv")
print(" - tableBI_final_acteurs.csv")
print(" - tableBI_final_realisateurs.csv")
print(" - tableBI_final_societes_production.csv")
print(" - tableBI_final_langues_originales.csv")

 OK - Exports générés dans : C:\Users\azerty\Desktop\Projet2 - Copie\data\processed\powerbi
 - movies_fr_final_powerbi.csv (table principale)
 - tableBI_final_genres.csv
 - tableBI_final_acteurs.csv
 - tableBI_final_realisateurs.csv
 - tableBI_final_societes_production.csv
 - tableBI_final_langues_originales.csv


In [4]:
# =====================================================
# TABLE BRIDGE : ZONE DE PRODUCTION
# (si "zone_de_production" est une valeur unique : 1 film -> 1 zone)
# =====================================================
table_zones = (
    main[["id_film", "zone_de_production"]]
    .dropna(subset=["zone_de_production"])
    .copy()
)

table_zones["id_film"] = table_zones["id_film"].astype(str).str.strip()
table_zones["zone_de_production"] = table_zones["zone_de_production"].astype("string").str.strip()
table_zones = table_zones[table_zones["zone_de_production"] != ""].drop_duplicates()

table_zones.to_csv(OUT_DIR / "tableBI_final_zones_production.csv", index=False, encoding="utf-8-sig")

print(" tableBI_final_zones_production.csv générée")

 tableBI_final_zones_production.csv générée
